# 21 · Análisis de errores y casos en prueba

Esta notebook analiza las predicciones de prueba que dejó la apertura única del conjunto de prueba (`src/abrir_prueba.py`, carpeta `modelo/salidas_s11/`). No entrena ni vuelve a evaluar modelos con otros umbrales: cada modelo conserva el umbral de decisión que se eligió por CSI en validación. Es un análisis descriptivo que responde dónde y cómo acierta o falla cada peldaño de la escalera de complejidad.

Las métricas por pares de prueba mostraron que ningún peldaño se distingue con significancia del umbral sobre el p10 de C13 en CSI, que la regresión logística ordena mejor que el umbral (AUC) y que las redes no superan a la logística. Las métricas por evento y los estratos pueden mostrar diferencias que las métricas por pares no ven, porque un aviso útil no necesita acertar todos los pares de un episodio.

**Alcance.**

| Sección | Qué | Modelos |
|-|-|-|
| 1 | Métricas por evento (POD y FAR por evento, anticipación) y estratos: noche, terciles de umbral, temporada, nube fría | Las 10 configuraciones: umbral, 3 logísticas y 6 redes (cada red por semilla y como media ± dispersión entre sus 3 semillas) |
| 2 | Métricas por estación y mapa (mínimo 10 episodios evaluables por estación) | Umbral C13, logística L2 y CNN plana L2 (semilla 42) |
| 3 | Falsas alarmas por intensidad en la ventana (incluida la banda 80–100 % del umbral) y por vecindad de tormenta | Los mismos tres |
| 4 | Casos de estudio: acierto diurno (25-abr-2025), omisión nocturna y falsa alarma | Los mismos tres |
| 5 | Coeficientes estandarizados de la regresión logística L2 | Logística L2 |

Los tres modelos de las secciones 2–5 son uno por peldaño: el umbral (peldaño 1), la logística con la historia más corta (peldaño 2) y el mejor modelo según la regla fijada antes de abrir prueba, la CNN plana L2 (peldaño 3). Para la red se usa la semilla 42, la que reproduce exactamente la búsqueda de hiperparámetros. La ResNet queda fuera de estas secciones porque en prueba no se distinguió de la CNN plana.

Los intervalos de confianza vienen del bootstrap por bloques de día pluviométrico (07:00 a 07:00 local) con B = 10,000 réplicas y semilla 42, los mismos bloques y pesos de la apertura de prueba. Las diferencias entre modelos en esta notebook son descriptivas: las comparaciones pareadas fijadas de antemano son las de la apertura de prueba.

In [ ]:
import sys, pathlib
_raiz = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / ".gitignore").exists())
sys.path.insert(0, str(_raiz))
import entorno

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ     = _raiz
RUTA_S11 = RAIZ / "modelo" / "salidas_s11"
RUTA_SAL = RAIZ / "modelo" / "salidas_21"
RUTA_SAL.mkdir(exist_ok=True)

sys.path.insert(0, str(RAIZ / "src"))
import evaluacion as ev
import analisis as an
from cargador import NG, cod10
from paleta import ROL, PALETA, aplicar_estilo
aplicar_estilo()

B, SEMILLA = ev.B_DEFECTO, 42          # los mismos bloques y pesos de la apertura de prueba
CFGS = an.configuraciones()
MODELOS = [m for mods in CFGS.values() for m in mods]
COLOR_RESULTADO = {"acierto": PALETA["wwu-green"], "omisión": ROL["positivo"],
                   "falsa alarma": PALETA["wi-ocher"], "correcto negativo": ROL["random_neg"]}

## 0. Datos

Se cargan las predicciones de prueba de los 22 modelos y sus umbrales de decisión (fijados en validación), los atributos de cada par (p10 de C13, intensidad en la ventana relativa al umbral, vecindad de tormenta) y los episodios de lluvia fuerte construidos con la misma definición que en la evaluación de validación (`int_t` ≥ umbral, unidos con pausas ≤ 30 min). La asignación de episodios debe reproducir exactamente las etiquetas: todo par positivo pertenece a un episodio y ningún negativo lo hace.

In [ ]:
base, P = an.cargar_predicciones(RUTA_S11, MODELOS)
UMBRAL = pd.read_csv(RUTA_S11 / "umbrales.csv", index_col=0)["umbral"].to_dict()
assert set(UMBRAL) == set(MODELOS)

atributos = pd.read_parquet(RAIZ / "datasets" / "negativos_atributos.parquet",
                            columns=["codigoestacion", "timestamp", "p10_c13", "razon", "n_est_pos", "n_est_activas"])
atributos["codigoestacion"] = cod10(atributos["codigoestacion"])
print("zonas horarias:", base.timestamp.dt.tz, atributos.timestamp.dt.tz)
pares = base.merge(atributos, on=["codigoestacion", "timestamp"], how="left", validate="one_to_one")
assert pares[["p10_c13", "razon"]].notna().all().all(), "Pares de prueba sin atributos"

etiquetas = pd.read_parquet(RAIZ / "data_ingestion" / "ground_truth" / "etiquetas_ground_truth.parquet")
lab = etiquetas.assign(codigoestacion=etiquetas.codigoestacion.astype(str))
lab = lab[lab.codigoestacion != NG]
umb = pd.read_csv(RAIZ / "data_ingestion" / "ground_truth" / "umbral_operativo_por_estacion.csv",
                  dtype={"codigoestacion": str}).set_index("codigoestacion")["umbral_mm_h"]
umb = umb[umb.index.isin(lab.codigoestacion.unique())]
episodios, fuertes = ev.episodios_lluvia(lab, umb)
pares["id_episodio"], n_dist = ev.asignar_episodio(pares, fuertes)
assert ((pares.id_episodio >= 0) == (pares.etiqueta == 1)).all(), "La asignación de episodios no reproduce las etiquetas"

grupo_umbral, cortes = ev.terciles_umbral(umb)
p10_par = pares.set_index(["codigoestacion", "timestamp"])["p10_c13"]
bloques = np.unique(ev.bloque_local(pares.timestamp))
pesos = ev.pesos_bootstrap(len(bloques), B=B, semilla=SEMILLA)

print(f"{len(pares):,} pares de prueba, {int(pares.etiqueta.sum()):,} positivos en "
      f"{pares.loc[pares.etiqueta == 1, 'id_episodio'].nunique():,} episodios; {pares.codigoestacion.nunique()} estaciones; "
      f"{len(bloques)} días pluviométricos; {(n_dist > 1).sum()} pares tocan dos episodios")
print(f"cortes de terciles de umbral: {np.round(cortes, 2)} mm/h")

## 1. Métricas por evento y estratos

Para cada modelo se construyen sus episodios evaluables y sus avisos (alertas consecutivas en la misma estación con pausas ≤ 30 min) y se calculan, en el total y en cada estrato, las métricas por pares (POD, FAR, CSI, ETS) y por evento (POD por evento = episodios avisados con al menos una alerta válida antes de su inicio / episodios evaluables; FAR por evento = avisos falsos / avisos; fracción de episodios avisados con al menos 1 h de anticipación). Los estratos son los de la evaluación de validación: noche (19:00–06:59 local), terciles del umbral de la estación, temporada y nube fría (p10 de C13 ≤ 228.2 K).

Cada red se reporta por semilla en `estratos_por_modelo.csv` (con su intervalo de confianza) y como media ± dispersión entre semillas en `estratos_por_configuracion.csv`. El cálculo recorre los 22 modelos sobre 2.4 millones de pares; toma varios minutos.

In [ ]:
t0 = time.time()
filas, anticipacion, EVENTOS = [], [], {}
for i, m in enumerate(MODELOS, 1):
    ep, av, ant, tab = an.evaluar_modelo(pares, P[m], UMBRAL[m], episodios, grupo_umbral, p10_par, pesos, bloques)
    filas.append(tab.assign(modelo=m))
    anticipacion.append({"modelo": m, **ant})
    if m in an.MODELOS_DETALLE:
        EVENTOS[m] = (ep, av)
    print(f"  {m}: {len(ep):,} episodios, {int(ep.avisado.sum())} avisados, {len(av):,} avisos "
          f"({i}/{len(MODELOS)}, {(time.time() - t0) / 60:.1f} min)", flush=True)

estratos = pd.concat(filas, ignore_index=True)
estratos.to_csv(RUTA_SAL / "estratos_por_modelo.csv", index=False)
anticipacion = pd.DataFrame(anticipacion)
anticipacion.to_csv(RUTA_SAL / "anticipacion_por_modelo.csv", index=False)
por_cfg = an.por_configuracion(estratos, CFGS)
por_cfg.to_csv(RUTA_SAL / "estratos_por_configuracion.csv", index=False)
poco = estratos.filter(like="_poco_confiable").any(axis=1)
print(f"\nIC poco confiables (> 1 % de réplicas indefinidas): {int(poco.sum())} filas de {len(estratos)}")

In [ ]:
cols = ["configuracion", "n_episodios", "pod_evento", "pod_evento_de_semillas", "far_evento", "far_evento_de_semillas",
        "frac_anticipacion_1h", "csi"]
total = por_cfg[por_cfg.particion == "total"][cols].set_index("configuracion")
ant_cfg = (anticipacion.assign(configuracion=anticipacion.modelo.map({m: c for c, ms in CFGS.items() for m in ms}))
           .groupby("configuracion", sort=False)[["n_avisados", "mediana_h"]].mean())
total.join(ant_cfg).round(3)

Intervalos de confianza por evento de los modelos de referencia y de cada semilla de la CNN plana L2 (total):

In [ ]:
sel = ["umbral_c13", "logistica_L2", "logistica_L3", "logistica_L4", "cnn_plana_L2_s42", "cnn_plana_L2_s43", "cnn_plana_L2_s44"]
t = estratos[(estratos.particion == "total") & estratos.modelo.isin(sel)].set_index("modelo").loc[sel]
pd.DataFrame({k: t[k].round(3).astype(str) + " [" + t[f"{k}_ic_inf"].round(3).astype(str) + ", "
              + t[f"{k}_ic_sup"].round(3).astype(str) + "]" for k in ("pod_evento", "far_evento", "csi")})

Estratos por configuración (media entre semillas para las redes). Se muestran el POD por evento y el CSI; las tablas completas, con FAR por evento, ETS e intervalos, están en los CSV.

In [ ]:
for part in ("noche", "nube_fria", "umbral", "temporada"):
    t = por_cfg[por_cfg.particion == part]
    tabla = t.pivot(index="configuracion", columns="estrato", values=["pod_evento", "csi"]).loc[list(CFGS)]
    n_ep = t.groupby("estrato").n_episodios.first()
    print(f"\n== {part} ==  episodios por estrato: {n_ep.to_dict()}")
    display(tabla.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
orden = list(CFGS)
for ax, (part, etiquetas_est) in zip(axes, [("noche", {False: "día", True: "noche"}),
                                            ("nube_fria", {False: "sin nube fría", True: "con nube fría"})]):
    t = por_cfg[por_cfg.particion == part]
    x = np.arange(len(orden))
    for k, (est_, nombre) in enumerate(etiquetas_est.items()):
        s = t[t.estrato.astype(str) == str(est_)].set_index("configuracion").reindex(orden)
        ax.bar(x + (k - 0.5) * 0.38, s.pod_evento, width=0.38, label=nombre,
               color=[ROL["serie"], ROL["positivo"]][k], yerr=s.pod_evento_de_semillas.fillna(0), capsize=2)
    ax.set_xticks(x, orden, rotation=60, ha="right"); ax.set_title(f"POD por evento: {part.replace('_', ' ')}")
    ax.legend(frameon=False); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("POD por evento (prueba)")
fig.tight_layout(); fig.savefig(RUTA_SAL / "img-pod_evento_estratos.png", dpi=150)

## 2. Métricas por estación

Solo en prueba y solo para los tres modelos de detalle. Una estación con menos de 10 episodios evaluables queda marcada como insuficiente, sin métricas, y se dibuja distinta en el mapa (regla fijada antes de ver resultados). Cada estación lleva su intervalo de confianza por bootstrap de días pluviométricos.

In [ ]:
por_estacion = []
for m in an.MODELOS_DETALLE:
    ep, av = EVENTOS[m]
    d = pares.assign(y=pares.etiqueta.to_numpy(), p=P[m])
    r = ev.evaluar_por_estrato(d, ep, av, UMBRAL[m], d.codigoestacion.to_numpy(), ep.codigoestacion.to_numpy(),
                               av.codigoestacion.to_numpy(), pesos, bloques, min_episodios=ev.MIN_EPISODIOS_ESTACION)
    por_estacion.append(an.a_numerico(r.rename(columns={"estrato": "codigoestacion"}).assign(modelo=m)))
por_estacion = pd.concat(por_estacion, ignore_index=True)
por_estacion.to_csv(RUTA_SAL / "por_estacion.csv", index=False)
r = por_estacion[por_estacion.modelo == "cnn_plana_L2_s42"]
print(f"{(~r.insuficiente).sum()} estaciones con ≥ {ev.MIN_EPISODIOS_ESTACION} episodios; {r.insuficiente.sum()} insuficientes")
por_estacion.pivot(index="codigoestacion", columns="modelo", values="pod_evento").describe().round(3)

In [ ]:
coords = pd.read_csv(RAIZ / "data_ingestion" / "ground_truth" / "anexo_A_estaciones.csv", dtype={"Código": str})
coords = coords.rename(columns={"Código": "codigoestacion", "Latitud (°N)": "lat", "Longitud (°)": "lon"})
coords["codigoestacion"] = coords.codigoestacion.str.zfill(10)
mapa = por_estacion.merge(coords[["codigoestacion", "lat", "lon"]], on="codigoestacion", how="left")
assert mapa.lat.notna().all(), "Estaciones sin coordenadas"

try:
    import geopandas as gpd, contextily as ctx
    g = gpd.GeoDataFrame(mapa, geometry=gpd.points_from_xy(mapa.lon, mapa.lat), crs="EPSG:4326").to_crs(3857)
    usar_mapa = True
except ImportError:
    g, usar_mapa = mapa.assign(x=mapa.lon, y=mapa.lat), False

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), sharex=True, sharey=True)
vmax = float(np.nanmax(mapa.pod_evento))
for ax, (m, nombre) in zip(axes, an.MODELOS_DETALLE.items()):
    s = g[g.modelo == m]
    ok, ins = s[~s.insuficiente], s[s.insuficiente]
    xs = (lambda d: d.geometry.x) if usar_mapa else (lambda d: d.x)
    ys = (lambda d: d.geometry.y) if usar_mapa else (lambda d: d.y)
    sc = ax.scatter(xs(ok), ys(ok), c=ok.pod_evento, cmap="viridis", vmin=0, vmax=vmax, s=70, edgecolor=ROL["linea_ref"], zorder=3)
    ax.scatter(xs(ins), ys(ins), facecolor="none", edgecolor=ROL["random_neg"], s=55, zorder=3, label="< 10 episodios")
    if usar_mapa:
        try:
            ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
        except Exception as e:
            print("sin mapa base:", e)
    ax.set_title(nombre); ax.set_xticks([]); ax.set_yticks([])
axes[0].legend(frameon=False, loc="lower left")
fig.colorbar(sc, ax=axes, shrink=0.8, label="POD por evento (prueba)")
fig.savefig(RUTA_SAL / "img-mapa_pod_evento_estacion.png", dpi=150, bbox_inches="tight")

## 3. Falsas alarmas por intensidad en la ventana y por vecindad de tormenta

Entre los negativos de prueba, ¿qué fracción recibe alerta según cuánta lluvia hubo en la ventana (razón entre la intensidad horaria máxima en la ventana y el umbral de la estación) y según si otra estación de la red tenía tormenta en el mismo instante? La banda 80–100 % del umbral está dentro de la incertidumbre del P95 (CV mediano 18.7 %): si los modelos alertan ahí mucho más que en el resto, parte de sus falsas alarmas son casi-aciertos que el umbral fijo no reconoce. Esa banda se dejó fuera del muestreo de negativos difíciles precisamente para analizarla aquí.

In [ ]:
falsas = pd.concat([an.tasa_alerta_negativos(pares, P[m], UMBRAL[m]).assign(modelo=nombre)
                    for m, nombre in an.MODELOS_DETALLE.items()], ignore_index=True)
falsas.to_csv(RUTA_SAL / "falsas_alarmas_por_grupo.csv", index=False)
orden_int = ["sin lluvia", "< 50 %", "50–80 %", "80–100 %"]
t = falsas.pivot_table(index=["criterio", "grupo"], columns="modelo", values="tasa_alerta")
n = falsas.drop_duplicates(["criterio", "grupo"]).set_index(["criterio", "grupo"]).n_negativos
display(t.join(n).round(4))

## 4. Casos de estudio

Tres días pluviométricos, uno por tipo de resultado, elegidos con una regla escrita antes de mirar las predicciones de esos días, aplicada a los episodios y avisos de la CNN plana L2 (semilla 42):

- **Acierto diurno:** 25-abr-2025, el evento convectivo que ya ilustra el parche (`10-goes_patches`) y el índice (`11-dataset_training`).
- **Omisión nocturna:** el día con más episodios que empiezan de noche (19:00–06:59 local) sin aviso.
- **Falsa alarma:** el día con más avisos falsos.

Empates: el día más temprano. Cada figura muestra, arriba, la temperatura de brillo de C13 en el instante del día con más pares positivos (o con más alertas falsas, en el caso de falsa alarma), con cada estación coloreada según el resultado de la CNN plana en ese instante; abajo, para cada modelo, la línea de tiempo del día en las estaciones involucradas: franjas donde el par es positivo y puntos donde el modelo alerta.

In [ ]:
casos = an.elegir_casos(*EVENTOS["cnn_plana_L2_s42"])
casos.to_csv(RUTA_SAL / "casos_estudio.csv", index=False)
casos

In [ ]:
from cache_imagenes import Cache
zs = pd.read_csv(RAIZ / "data_ingestion" / "satellite" / "normalizacion_zscore.csv")
cache = Cache(RAIZ / "data" / "cache", zs, en_memoria=False)
IB13 = cache.bandas.index("CMI_C13")

def c13(t_utc):
    fila = int(cache.fila_de[pd.Timestamp(t_utc).tz_convert(None)])
    return cache.img[fila, IB13].astype(float) * float(cache.scale[IB13]) + float(cache.offset[IB13])

def figura_caso(caso, dia, max_estaciones=20):
    ini, fin = an.ventana_dia(dia)
    d = pares[(pares.timestamp >= ini) & (pares.timestamp < fin)].copy()
    alertas = {m: P[m][d.index] >= UMBRAL[m] for m in an.MODELOS_DETALLE}
    y = d.etiqueta.to_numpy() == 1
    a_cnn = alertas["cnn_plana_L2_s42"]
    clave = (a_cnn & ~y) if caso == "falsa alarma" else y
    t_star = d.loc[clave].groupby("timestamp").size().idxmax() if clave.any() else d.timestamp.iloc[0]
    # estaciones involucradas: con algún positivo o alguna alerta de los tres modelos, las más activas primero
    act = d.assign(pos=y, al=np.column_stack(list(alertas.values())).any(1)).groupby("codigoestacion")[["pos", "al"]].sum()
    est = act[(act.pos + act.al) > 0].sort_values(["pos", "al"], ascending=False).index[:max_estaciones].tolist()

    fig = plt.figure(figsize=(13, 10))
    gs = fig.add_gridspec(4, 2, width_ratios=[1, 2.3], height_ratios=[1, 1, 1, 0.05], hspace=0.5, wspace=0.15)
    ax0 = fig.add_subplot(gs[0:3, 0])
    im = ax0.imshow(c13(t_star), cmap="Greys", origin="upper")
    en_t = (d.timestamp == t_star).to_numpy()
    res = an.resultado_par(y[en_t], a_cnn[en_t])
    for cod, r_ in zip(d.codigoestacion[en_t], res):
        fc, cc = cache.pos[int(cod)]
        ax0.scatter(cc, fc, color=COLOR_RESULTADO[r_], s=45, edgecolor="white", linewidth=0.6, zorder=3)
    for r_, col in COLOR_RESULTADO.items():
        ax0.scatter([], [], color=col, label=r_, s=40)
    ax0.legend(frameon=False, fontsize=8, ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.12))
    ax0.set_title(f"C13 a las {t_star.tz_convert(ev.ZONA_LOCAL):%H:%M} local\n(resultado de la CNN plana L2)", fontsize=10)
    ax0.set_xticks([]); ax0.set_yticks([])
    fig.colorbar(im, ax=ax0, orientation="horizontal", fraction=0.05, pad=0.03, label="temperatura de brillo (K)")

    hora = lambda s: (s.dt.tz_convert(ev.ZONA_LOCAL) - (ini.tz_convert(ev.ZONA_LOCAL))) / pd.Timedelta(hours=1)
    pos_y = {c: i for i, c in enumerate(est)}
    for k, (m, nombre) in enumerate(an.MODELOS_DETALLE.items()):
        ax = fig.add_subplot(gs[k, 1])
        sub = d[d.codigoestacion.isin(est)]
        h, yy = hora(sub.timestamp), sub.codigoestacion.map(pos_y)
        pos = sub.etiqueta.to_numpy() == 1
        ax.scatter(h[pos], yy[pos], marker="s", s=14, color=ROL["positivo"], alpha=0.35, linewidth=0, label="par positivo")
        al = alertas[m][d.codigoestacion.isin(est).to_numpy()]
        ax.scatter(h[al], yy[al], marker="o", s=9, color=ROL["serie"], label="alerta")
        ax.axvline(hora(pd.Series([t_star])).iloc[0], color=ROL["linea_ref"], lw=0.8, ls="--")
        ax.set_yticks(range(len(est)), [c[-4:] for c in est], fontsize=6)
        ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 3), [f"{(7 + x) % 24:02d}" for x in range(0, 25, 3)])
        ax.set_title(nombre, fontsize=10, loc="left")
        if k == 0:
            ax.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
    ax.set_xlabel("hora local (día pluviométrico 07:00 → 07:00)")
    fig.suptitle(f"{caso}: {dia}", fontsize=12)
    nombre_archivo = RUTA_SAL / f"img-caso_{caso.replace(' ', '_')}.png"
    fig.savefig(nombre_archivo, dpi=150, bbox_inches="tight")
    return fig

for _, c in casos.iterrows():
    figura_caso(c.caso, c.dia)

## 5. Coeficientes de la regresión logística L2

La logística es el modelo que mejor ordena los pares de prueba (AUC 0.79). Sus coeficientes, estimados sobre variables estandarizadas, indican cuánto cambia el logaritmo de las chances de lluvia fuerte por un desvío estándar de cada estadística del parche, con las demás fijas. Se reajusta igual que en la apertura de prueba (mismos pares de entrenamiento ponderados 1:2:2, C = 10; es determinista) y se verifica que reproduce las predicciones de prueba guardadas. Las variables están correlacionadas (p. ej. p10 y p50 de una misma banda, o la misma banda en dos instantes), de modo que un coeficiente aislado no mide la importancia de una variable: se lee el patrón por banda y por instante, no cada valor por separado.

In [ ]:
import referencias as rf
from cargador import L_COMUN, preparar_pares
indice = pd.read_parquet(RAIZ / "datasets" / "indice_completo.parquet")
atr = pd.read_parquet(RAIZ / "datasets" / "negativos_atributos.parquet")
est_parche, meta = rf.leer_estadisticas(RAIZ / "data" / "cache" / "estadisticas_parche")
codigos, bandas = meta["estaciones"], meta["bandas"]

tr = preparar_pares(indice, atr, cache, "train", 2)
log_l2 = rf.ajustar(rf.variables_pares(tr, est_parche, codigos, 2), tr.etiqueta.to_numpy(), rf.pesos_muestreo(tr.bolsa), C=10.0)
te = preparar_pares(indice, atr, cache, "test", L_COMUN)
p_te = rf.predecir(log_l2, rf.variables_pares(te, est_parche, codigos, L_COMUN)[:, -2 * len(bandas) * 4:])
control = te[["codigoestacion", "timestamp"]].assign(p_nuevo=p_te).merge(
    pares[["codigoestacion", "timestamp"]].assign(p_s11=P["logistica_L2"]), on=["codigoestacion", "timestamp"], validate="one_to_one")
dif = np.abs(control.p_nuevo - control.p_s11).max()
print(f"máxima diferencia con las predicciones de prueba de la apertura: {dif:.2e}")
assert dif < 1e-4, "La logística reajustada no reproduce la de la apertura de prueba"

coef = an.coeficientes_logistica(log_l2, bandas, 2)
coef.to_csv(RUTA_SAL / "coeficientes_logistica_L2.csv", index=False)
coef.assign(abs_coef=coef.coef.abs()).groupby("banda").abs_coef.sum().sort_values(ascending=False).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
lim = float(coef.coef.abs().max())
for ax, inst in zip(axes, coef.instante.unique()):
    m = coef[coef.instante == inst].pivot(index="banda", columns="estadistica", values="coef")[["p10", "p50", "p90", "desvio"]]
    im = ax.imshow(m.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_xticks(range(4), m.columns); ax.set_yticks(range(len(m)), [b.replace("CMI_", "") for b in m.index])
    for (i, j), v in np.ndenumerate(m.to_numpy()):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"instante {inst}")
fig.colorbar(im, ax=axes, shrink=0.85, label="coeficiente estandarizado")
fig.savefig(RUTA_SAL / "img-coeficientes_logistica_L2.png", dpi=150, bbox_inches="tight")

## 6. Lectura

*(Se redacta con los resultados de las secciones 1–5.)*